<a href="https://colab.research.google.com/github/TechTinkerKetki/curriculum_bot_iiti/blob/main/03_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [114]:
!pip install -q google-generativeai faiss-cpu


In [ ]:
from pathlib import Path
import json
import numpy as np

In [117]:
import getpass
import os

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")


KeyboardInterrupt: Interrupted by user

In [86]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GEMINI_API_KEY"])


In [ ]:
DATA_DIR = Path("data/processed")

CHUNKS_PATH = DATA_DIR / "chunks.json"
FAISS_PATH = DATA_DIR / "faiss.index"


In [87]:
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Loaded chunks:", len(chunks))


Chunks loaded: 184


In [88]:
def retrieve(query, k=5):
    q_emb = genai.embed_content(
        model="models/text-embedding-004",
        content=query
    )["embedding"]

    q_emb = np.array(q_emb).reshape(1, -1)

    distances, indices = index.search(q_emb, k)

    # return [chunks[i] for i in indices[0]]
    return [
    c for c in (chunks[i] for i in indices[0])
    if "Curriculum of" not in c["content"][:50]
]


In [89]:
def format_context(chunks):
    formatted = []
    for c in chunks:
        page = c["metadata"].get("page", "N/A")
        formatted.append(
            f"[Page {page}]\n{c['content']}"
        )
    return "\n\n---\n\n".join(formatted)



In [90]:
def build_prompt(retrieved_chunks, question):
    context = format_context(retrieved_chunks)

    return f"""
You are an academic assistant for IIT Indore.

Answer the question ONLY using the sources below.
If multiple curriculum versions are present, clearly separate them by academic year.
If the answer is not present, say:
"Not found in the provided curriculum."
When stating facts, cite them using page numbers
in square brackets, e.g. [Page 42].

Sources:
{context}

Question:
{question}

Answer :
"""


In [91]:
model_llm = genai.GenerativeModel("gemini-2.5-flash")

def answer_question(question, k=5):
    retrieved = retrieve(question, k)
    prompt = build_prompt(retrieved, question)
    response = model_llm.generate_content(prompt)
    return response.text


In [113]:
import faiss
import google.generativeai as genai # Corrected import back to google.generativeai

# Redefine model_llm to use the requested model
model_llm = genai.GenerativeModel("gemini-2.5-flash") # Model name remains as requested

print("Generating embeddings for chunks...")
chunk_contents = [c["content"] for c in chunks]
embeddings = genai.embed_content(
    model="models/text-embedding-004",
    content=chunk_contents
)["embedding"]
embeddings_np = np.array(embeddings).astype('float32')

print(f"Embeddings generated with shape: {embeddings_np.shape}")

d = embeddings_np.shape[1] # dimension of embeddings
index = faiss.IndexFlatL2(d)
index.add(embeddings_np)
faiss.write_index(index, str(FAISS_PATH))
print("Saved FAISS index to:", FAISS_PATH)




Generating embeddings for chunks...
Embeddings generated with shape: (184, 768)


NameError: name 'FAISS_PATH' is not defined

In [104]:
print(answer_question("What are the courses in Semester III?", k=4))

**Curriculum of 2nd Year B. Tech. (CSE) (From AY 2011-12 to AY 2013-14)** [Page 13]
*   HS 201 / HS 203 / HS 205 / HS 207 Understanding Philosophy / Psychology / Sociology / French Language – I
*   MA 201 Mathematics-III (Complex Analysis and Differential Equations-II)
*   CS 201 Discrete Mathematical Structures
*   CS 203 Data Structures and Algorithms
*   CS 205 Abstraction and Paradigms for Programming
*   CS 253 Data Structures and Algorithms Lab
*   CS 255 Abstraction and Paradigms for Programming Lab
*   IC 211 Experimental Engineering Lab

**Curriculum of 2nd Year B. Tech. (CSE) (From AY 2014-15 onwards to AY 2023-24)** [Page 13]
*   ZZ XXX Course-I for Minor Program
*   MA 203 Complex Analysis and Differential Equations-II
*   CS 201 Discrete Mathematical Structures
*   CS 203 Data Structures and Algorithms
*   CS 207 Data Base & Information Systems
*   CS 253 Data Structures and Algorithms Lab
*   CS 257 Data Base & Information Systems Lab
*   IC 211 Experimental Engineering L

In [105]:
print(answer_question("what is the course title of  cs451?", k=1))

**From AY 2010-11 to 2013-14:**
The course title of CS 451 is Soft Computing Lab [Page 148].


In [106]:
print(answer_question("what are the reference books for CS103?", k=1))

The reference books for CS103 are:
1.  G. Dromey, How to Solve It by Computer, Prentice-Hall, Inc., Upper Saddle River, NJ, 1982 [Page 37].
2.  Coohoon and Davidson, C++ Program Design: An introduction to Programming and Object-Oriented Design (3rd edition), Tata McGraw Hill, New Delhi, 2003 [Page 37].
3.  Yashwant Kanetkar, Let us C. Allied Publishers, 1998 [Page 37].
4.  G. Polya, How to Solve It (2nd ed.), Doubleday and co. (1957) [Page 37].
5.  The Java Tutorial, Sun Microsystems. Addison-Wesley, 1999 [Page 37].


In [107]:
print(answer_question("what are the minor courses for AY 2024 onwards?", k=4))

**Academic Year 2024-25 onwards (For all UG batches admitted in and after AY 2023-24)**

For the 3rd semester (Minor1), the minor courses are:
*   **Minor Program in BSBE**: BSE 201: Biophysics [Page 33]
*   **Minor Program in Chemistry**: CH 201: Molecules that Change the World [Page 33]
*   **Minor Program in Economics**: HS 209: Intermediate Microeconomics [Page 33]
*   **Minor Program in Liberal Arts**: HS 211: German Literature and Culture Studies, HS 212: History of India after Independence, 1947-2000, HS 203: Psychology [Page 33]
*   **Minor Program in Astronomy**: AA 201: Introduction to Astronomy [Page 33]


In [108]:
print(answer_question("what are the credits for ME204?", k=1))

Not found in the provided curriculum.
